# Brent · `description` (web: "Description")

**Goal:** understand the data, so that a chat bot can answer any question about it
more easily and correctly.

Brent only (golden rule): every query carries the location-based Brent filter.
Data comes from the **live API**.

Workflow: add a step here → discuss the findings → decide whether it belongs in
`knowledge/brent.md` → add it.

In [1]:
from pathlib import Path

import httpx
import pandas as pd
from dotenv import dotenv_values

PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
env = dotenv_values(PROJECT_ROOT / ".env")

BASE_URL = "https://planningdata.london.gov.uk/api-guest"
HEADERS = {"X-API-AllowRequest": env["DATAHUB_ALLOW_REQUEST"]}


def brent_filter() -> dict:
    # Location-based scope: borough is Brent, or borough is missing and Brent decided it.
    return {
        "bool": {
            "should": [
                {"term": {"borough.raw": "Brent"}},
                {"bool": {
                    "filter": [{"term": {"lpa_name.raw": "Brent"}}],
                    "must_not": [{"exists": {"field": "borough.raw"}}],
                }},
            ],
            "minimum_should_match": 1,
        }
    }


def search(body: dict) -> dict:
    response = httpx.post(f"{BASE_URL}/applications/_search", headers=HEADERS, json=body, timeout=60)
    response.raise_for_status()
    return response.json()

## Step 1 · Nulls and empty values

Fetch `description` for every Brent application (paged with `search_after`, sorted by
`id`, because there are more than 10,000). Then count:
- **null**: no value at all;
- **empty or blank**: a string with nothing in it, or only whitespace.

In [2]:
rows, after = [], None
while True:
    body = {"size": 5000, "query": brent_filter(), "_source": ["id", "description"], "sort": [{"id": "asc"}]}
    if after:
        body["search_after"] = after
    page = search(body)["hits"]["hits"]
    if not page:
        break
    rows += [h["_source"] for h in page]
    after = page[-1]["sort"]

df = pd.DataFrame(rows).reindex(columns=["id", "description"])
print("Brent applications:", len(df))

Brent applications: 36952


In [3]:
value = df["description"]
is_null = value.isna()
is_blank = value.notna() & (value.astype(str).str.strip() == "")

summary = pd.Series({
    "null": int(is_null.sum()),
    "empty or blank": int(is_blank.sum()),
    "filled": int((~is_null & ~is_blank).sum()),
})
summary["total"] = summary.sum()
summary.to_frame("count").assign(**{"share_%": lambda t: (100 * t["count"] / len(df)).round(2)})

,count,share_%
null,408,1.1
empty or blank,0,0.0
filled,36544,98.9
total,36952,100.0


### Findings · Step 1

Live API, 2026-11-09, 36,952 Brent applications:

| | Count | Share |
|---|---|---|
| null | 408 | 1.10% |
| empty or blank | 0 | 0% |
| filled | 36,544 | 98.90% |

- **Almost every record has a description** (98.9%).
- **408 are null.** That is close to the 407 empty records (open question 1), which are
  known to have no description; not checked yet.

## Step 2 · Do the values match the schema type?

The schema (sheet `applications`) gives `description` the type `string`, with no list of
allowed values. The live index mapping stores it as `text` (full-text only, no `.raw`
keyword field). Check the Python type of every value in `df` from Step 1: a JSON string
comes back as `str`; anything else (number, list, object) would not match.

In [4]:
df["description"].map(lambda v: type(v).__name__).value_counts().to_frame("count")

,count
description,
str,36544
float,408


### Findings · Step 2

Live API, 2026-11-09:

| Python type | Count |
|---|---|
| `str` | 36,544 |
| `float` | 408 |

- **Every filled value is a string**, as the schema says.
- The 408 `float` are the 408 nulls from Step 1: pandas stores a missing value as `NaN`,
  which is a float. They are not numbers in the data.
- `description` is `text` only in the index (no `.raw`), so exact-value counts need the
  values fetched, as done here; an Elasticsearch `terms` aggregation won't work on it.

## Step 3 · Most common descriptions

Count identical descriptions in `df` (exact text) and show the 30 most common.

In [5]:
top = df["description"].value_counts().head(30).to_frame("count")
print("distinct descriptions:", df["description"].nunique(), "| filled:", df["description"].notna().sum())
pd.set_option("display.max_colwidth", 150)
top

distinct descriptions: 31315 | filled: 36544


,count
description,
"Prior approval for a single storey rear extension to dwellinghouse, in metres:Extending beyond the rear wall of the original house - 6mMaximum height - 3mEaves height - 3m",347
No description provided by LDMS,227
"Prior approval for a single storey rear extension to dwellinghouse, in metres:Extending beyond the rear wall of the original house - 6mMaximum height - 4mEaves height - 3m",173
Certificate of lawfulness for proposed single storey rear extension to dwellinghouse,144
Proposed single storey rear extension to dwellinghouse,102
"Prior approval for a single storey rear extension to dwellinghouse, in metres:Extending beyond the rear wall of the original house - 6mMaximum height - 3.2mEaves height - 3m",85
Certificate of lawfulness for proposed single storey rear extension to dwellinghouse.,79
"Prior approval for a single storey rear extension to dwellinghouse, in metres:Extending beyond the rear wall of the original house - 6mMaximum height - 3.4mEaves height - 3m",78
"Prior approval for a single storey rear extension to dwellinghouse, in metres:Extending beyond the rear wall of the original house - 6mMaximum height - 3mEaves height - 2.9m",76


### Findings · Step 3

Live API, 2026-11-09: 36,544 filled descriptions, 31,315 distinct (86%).

- **Mostly unique, real text.** The most common description is used only 347 times.
- **One placeholder:** `No description provided by LDMS` (227), the second most common.
  It says nothing about the development (seen before on the 25 early-1987 records).
- **The rest of the top 30 are real, repeated wording** for common small works: prior
  approval for a single storey rear extension (a template with the measurements filled
  in), certificates of lawfulness, rear extensions, outbuildings, solar panels.
- **The same thing is written in slightly different ways:** "single storey" / "single-storey",
  "dwellinghouse" / "the dwellinghouse", with or without a full stop. Exact matching
  would treat these as different descriptions.
- **Words run together in the prior-approval template:** e.g. `- 6mMaximum heig...`.
  A line break was probably lost, so `6m` and `Maximum` are joined.

## Step 4 · `application_type` and `application_type_full` where description is null

Fetch the records with no `description` and count their `application_type` and
`application_type_full` together.

In [6]:
body = {
    "size": 1000,
    "query": {"bool": {
        "filter": [brent_filter()],
        "must_not": [{"exists": {"field": "description"}}],
    }},
    "_source": ["id", "application_type", "application_type_full"],
}
hits = search(body)["hits"]
no_description = pd.DataFrame([h["_source"] for h in hits["hits"]]).reindex(
    columns=["id", "application_type", "application_type_full"]
)
print("no description:", hits["total"]["value"], "| fetched:", len(no_description))

no_description.value_counts(["application_type", "application_type_full"], dropna=False).to_frame("count")

no description: 408 | fetched: 408


count
application_type application_type_full                                         
All Other        NaN                                                        274
Householder      NaN                                                        107
Prior Approval   NaN                                                         12
All Other        Full planning permission                                     7
Householder      Householder planning permission                              6
All Other        Lawful development: Proposed use                             1
                 Approval of details reserved by a condition (discharge)      1

### Findings · Step 4

Live API, 2026-11-09, the 408 records with no `description`:

| `application_type` | `application_type_full` | Count |
|---|---|---|
| All Other | null | 274 |
| Householder | null | 107 |
| Prior Approval | null | 12 |
| All Other | Full planning permission | 7 |
| Householder | Householder planning permission | 6 |
| All Other | Lawful development: Proposed use | 1 |
| All Other | Approval of details reserved by a condition (discharge) | 1 |

- **Every one has an `application_type`**, so the broad group is always known.
- **393 of 408 (96%) also have no `application_type_full`.** For these, the broad group is
  the only thing known about what was applied for.
- 15 have a detailed type, and in all of them the two fields agree.

## Step 5 · Overlap of the 408 with the 407 empty records

- **408**: no `description` (Step 1).
- **407**: no `status`, no `decision` and no `decision_date` (the empty records, open question 1).

Uses `_count` with the Brent filter. The records only in the 408 are listed.

In [7]:
def count(*extra: dict) -> int:
    body = {"query": {"bool": {"filter": [brent_filter(), *extra]}}}
    response = httpx.post(f"{BASE_URL}/applications/_count", headers=HEADERS, json=body, timeout=60)
    response.raise_for_status()
    return response.json()["count"]


def missing(field: str) -> dict:
    return {"bool": {"must_not": [{"exists": {"field": field}}]}}


the_407 = [missing("status"), missing("decision"), missing("decision_date")]

display(pd.Series({
    "the 408": count(missing("description")),
    "the 407": count(*the_407),
    "in both": count(missing("description"), *the_407),
}).to_frame("count"))

only_408 = search({
    "size": 100,
    "query": {"bool": {
        "filter": [brent_filter(), missing("description")],
        "must_not": [{"bool": {"filter": the_407}}],
    }},
    "_source": ["id", "status", "decision", "decision_date", "valid_date", "application_type", "bo_system"],
})["hits"]["hits"]
pd.DataFrame([h["_source"] for h in only_408])

,count
the 408,408
the 407,407
in both,407


,bo_system,decision_date,valid_date,decision,application_type,id,status
0,CSV Data Import,None,28/06/2019,Withdrawn,Prior Approval,Brent-19_2377,Withdrawn


### Findings · Step 5

Live API, 2026-11-09:

| Group | Count |
|---|---|
| in both | 407 |
| only in the 408 | 1 |
| only in the 407 | 0 |

- **All 407 empty records have no description.** No description is one more sign that
  they are empty records.
- **Only 1 real record has no description:** `Brent-19_2377`, a Prior Approval,
  valid 28/06/2019, Withdrawn (status and decision), no decision date, CSV Data Import.